# **Multilevel and marginal modeling, a case study with the NHANES data**


This notebook is a counterpart to our introductory regression modeling case study for independent data using the [NHANES](https://www.cdc.gov/nchs/nhanes/index.htm) data.  Here we will build on the basic linear and logistic regression approaches discussed in that notebook.  We will be exploring the use of some more advanced regression approaches that can be used for data that are statistically dependent.

Note that some of the models illustrated in this notebook may take a few minutes to fit.

We begin by importing the libraries that we will be using.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import statsmodels.api as sm
import numpy as np

Data can be *dependent*, or have a *mutilevel structure* for a number of different reasons.  Arguably, most datasets encountered in practice exhibit some form of dependence, and it is independent data, not dependent data, should be treated as the exceptional case. Here we will reconsider the NHANES data from the perspective of dependence, focusing in particular on dependence in the data that arises due to *clustering* which we will define below.

First, we read the data from a csv file to a Pandas dataframe, as we have done before.  We remove all rows of data with missing values in the variables of interest (there are more sophisticated approaches for handling missing data that generally will give better results, but for simplicity we do not use them here).

Note that in addition to the demographic and health-related variables that we have used before, we also retain two variables here [SDMVSTRA](https://wwwn.cdc.gov/Nchs/Nhanes/2015-2016/DEMO_I.htm#SDMVSTRA) and [SDMVPSU](https://wwwn.cdc.gov/Nchs/Nhanes/2015-2016/DEMO_I.htm#SDMVPSU) that will be used below to define the clustering structure in these data.

In [ ]:
# Read the data file
da = pd.read_csv("/content/sample_data/nhanes_2015_2016.csv")

# Drop unused columns, drop rows with any missing values.
vars = ["BPXSY1", "RIDAGEYR", "RIAGENDR", "RIDRETH1", "DMDEDUC2", "BMXBMI",
        "SMQ020", "SDMVSTRA", "SDMVPSU"]
da = da[vars].dropna()

## **Introduction to clustered data**

One common reason that data are statistically dependent is that the data values were collected as a "cluster sample".  This essentially means that the population of interest was first partitioned into groups, then a limited number of these groups were somehow selected, and finally a limited number of individuals were selected from each of the sampled groups. This is the protocol used to collect the NHANES data.  Since NHANES involves physical examinations, it is not practical to select a random sample from the entire US population, as this would involve conducting the examinations at thousands of dispersed locations.  By utilizing cluster sampling, the NHANES staff can set up an examination center in each selected community, and assess many subjects at each center.

Cluster sampling is not the only reason that dependence may exist between observations in a dataset.  For example, many studies are *longitudinal*, meaning that each subject is assessed on multiple occasions.  In this setting, we would expect these repeated measurements to be correlated.  Since NHANES is a cluster sample, but is not a longitudinal study, we will focus on cluster sampling here to illustrate multilevel modeling.

In any cluster sample, it is likely that observations within the same cluster are more similar than observations in different clusters.  For example, in NHANES the clusters are geographic, and each sample community is somewhat unique in terms of demography, climate, socio-economic status, and lifestyle.  When we have clustered data, it is very importantto account for this statistical dependence in the analysis.

## **Clustering structure in NHANES**

The detailed process of collecting data for a study like NHANES is very complex, so we will simplify things here (more
details about the NHANES sampling plan can be found [here](https://wwwn.cdc.gov/nchs/nhanes/analyticguidelines.aspx) but are not needed for this course). Roughly speaking, in NHANES the data are collected by selecting a limited number of counties in the US, then selecting subregions of these counties, then selecting people within these subregions.  Since counties are geographically constrained, it is expected that people within a county are more similar to each other than they are to people in other counties.

If we could obtain the county identifier where each NHANES participant resides, we could directly incorporate this information into our analysis.  However for privacy reasons this information is not released with the data. Instead, we have access to  **"masked variance units" (MVUs), which are formed by combining subregions of different counties into artificial groups that are not geographically contiguous**.  While the MVUs are not the actual clusters of the survey, and are not truly contiguous geographic regions, they are deliberately selected to mimic these things, while minimizing the risk that a subject can be "unmasked" in the data.  For the remainder of this notebook, we will treat the MVUs as clusters, and explore the extent to which they induce correlations in some of the NHANES variables that we have been studying.

The MVU identifiers can be obtained by combining the `SDMVSTRA` and `SDMVPSU` identifiers, which we do next:

In [ ]:
# Each person is assigned to an artificial survey cluster

da["group"] = 10*da.SDMVSTRA + da.SDMVPSU

## **Intraclass correlation**

Similarity among observations within a cluster can be measured using a statistic called the [intraclass correlation coefficient](https://en.wikipedia.org/wiki/Intraclass_correlation), or ICC.  This is a distinct form of correlation from the more well-known [Pearson's correlation](https://en.wikipedia.org/wiki/Pearson_correlation_coefficient).  The ICC takes on values from 0 to 1, with 1 corresponding to "perfect clustering" -- the values within a cluster are identical, and 0 corresponding to "perfect independence" -- the mean value within each cluster is identical across all the clusters.

We can assess ICC using two regression techniques, *marginal regression*, and *multilevel regression*.  We will start by using a technique called "Generalized Estimating Equations" (GEE) to fit marginal linear models, and to estimate the ICC for the NHANES clusters.

We will first look at the ICC for systolic blood pressure:

In [ ]:
#Exchangeable: Any two people inside the same group have the same correlation.
#This code creates NHANES cluster IDs, fits a GEE model for systolic blood pressure,
#and estimates how correlated blood pressure values are among people in the same cluster.
model = sm.GEE.from_formula("BPXSY1 ~ 1", groups="group",
           cov_struct=sm.cov_struct.Exchangeable(), data=da)
result = model.fit()
print(result.cov_struct.summary())

The correlation between two observations in the same cluster is 0.030


The estimated ICC is 0.03. This is small, but it still matters.

An ICC is a type of correlation, but it should not be interpreted exactly like a Pearson correlation. A Pearson correlation of 0.03 would usually mean almost no relationship. However, an ICC of 0.03 can still be meaningful because it shows that people within the same cluster are slightly more similar to each other than to people in other clusters.

In simple terms:

The clustering effect is weak, but not zero.

To better understand how much clustering exists in the dataset, we calculate ICC values for several variables used in the analysis, including both outcomes and predictors. This helps us see which variables are more affected by the NHANES cluster structure.

In [ ]:
# Recode smoking to a simple binary variable, smq becomes a simple binary variable:
# smq = 1 \text{ if smoker} smq = 0 \text{ if non-smoker}

#Variable Meaning
#BPXSY1: systolic blood pressure
#RIDAGEYR: age
#BMXBMI: body mass index
#smq:smoking status
#SDMVSTRA: survey stratum
da["smq"] = da.SMQ020.replace({2: 0, 7: np.nan, 9: np.nan})

# For each variable, the model uses: v + " ~ 1" Estimate only the average value of that variable.
# How similar are people inside the same cluster for this variable?
# So the model asks:
# After estimating the average value, are observations within the same group still correlated?

for v in ["BPXSY1", "RIDAGEYR", "BMXBMI", "smq", "SDMVSTRA"]:
    model = sm.GEE.from_formula(v + " ~ 1", groups="group",
           cov_struct=sm.cov_struct.Exchangeable(), data=da)
    result = model.fit()
    print(v, result.cov_struct.summary())

BPXSY1 The correlation between two observations in the same cluster is 0.030
RIDAGEYR The correlation between two observations in the same cluster is 0.035
BMXBMI The correlation between two observations in the same cluster is 0.039
smq The correlation between two observations in the same cluster is 0.026
SDMVSTRA The correlation between two observations in the same cluster is 0.959


The values are generally similar to what we saw for blood pressure,
except for `SDMVSTRA`, which is one component of the cluster
definition itself, and therefore has a very high ICC.

To illustrate that the ICC values shown above are not consistent with
a complete absence of dependence, we simulate 10 sets of random data
and calculate the ICC value for each set:

In [ ]:
for k in range(10):
    da["noise"] = np.random.normal(size=da.shape[0])
    model = sm.GEE.from_formula("noise ~ 1", groups="group",
           cov_struct=sm.cov_struct.Exchangeable(), data=da)
    result = model.fit()
    print(v, result.cov_struct.summary())

SDMVSTRA The correlation between two observations in the same cluster is 0.000
SDMVSTRA The correlation between two observations in the same cluster is 0.000
SDMVSTRA The correlation between two observations in the same cluster is -0.000
SDMVSTRA The correlation between two observations in the same cluster is -0.001
SDMVSTRA The correlation between two observations in the same cluster is -0.002
SDMVSTRA The correlation between two observations in the same cluster is 0.003
SDMVSTRA The correlation between two observations in the same cluster is -0.000
SDMVSTRA The correlation between two observations in the same cluster is -0.001
SDMVSTRA The correlation between two observations in the same cluster is 0.000
SDMVSTRA The correlation between two observations in the same cluster is -0.002


The simulated noise represents data where people are completely independent.

So if the NHANES data were also independent, their ICC values should be close to zero, like:

0.000,\ 0.001,\ -0.001

But the NHANES ICC values are clearly bigger.

Therefore:

Even though the NHANES ICC values look small, they are still meaningful because they are much larger than what we would expect under complete independence.

Very simple final idea

Random noise ICC:

\approx 0

NHANES ICC:

\approx 0.02 \text{ or } 0.03

Conclusion:

**The NHANES observations are not fully independent; people within the same cluster are slightly more similar to each other.**

## **Conditional intraclass correlation**

The ICC's studied above were *marginal*, in the sense that we were looking at whether, say, the SBP values were more similar within versus between clusters.  To the extent that such "cluster effects" are found, it may be largely explained by demographic differences among the clusters.  For example, we know from our previous analyses with the NHANES data that older people have higher SBP than younger people.  Also, some clusters may contain a slightly older or younger set of people than others.  Thus, by controlling for age, we might anticipate that the ICC will become smaller.  This is shown in the next analysis:

In [ ]:
model = sm.GEE.from_formula("BPXSY1 ~ RIDAGEYR", groups="group",
           cov_struct=sm.cov_struct.Exchangeable(), data=da)
result = model.fit()
print(result.cov_struct.summary())

The correlation between two observations in the same cluster is 0.019


The ICC for SBP drops from 0.03 to 0.02.  We can now assess whether it
drops even further when we add additional covariates that we know to
be predictive of blood pressure.

In [ ]:
# Create a labeled version of the gender variable
da["RIAGENDRx"] = da.RIAGENDR.replace({1: "Male", 2: "Female"})

model = sm.GEE.from_formula("BPXSY1 ~ RIDAGEYR + RIAGENDRx + BMXBMI + C(RIDRETH1)",
           groups="group",
           cov_struct=sm.cov_struct.Exchangeable(), data=da)
result = model.fit()
print(result.cov_struct.summary())

The correlation between two observations in the same cluster is 0.013


The variable [RIDRETH1](https://wwwn.cdc.gov/Nchs/Nhanes/2015-2016/DEMO_I.htm#RIDRETH1) is a categorical variable containing 5 levels of race/ethnicity information.  Since NHANES categorical variables are coded numerically, Statsmodels would have no way of knowing that these are codes and not quantitative data, thus we must use the `C()` syntax in the formula above to force this variable to be treated as being categorical.  *We see here that the ICC has further reduced, to 0.013, due to controlling for these additional factors including ethnicity.*

## **Marginal linear models with dependent data**

Above we focused on quantifying the dependence induced by clustering.
By understanding the clustering structure, we have gained additional
insight about the data that complements our understanding of the mean
structure.  Another facet of working with dependent data is that while
the mean structure (i.e. the regression coefficients) can be estimated
without considering the dependence structure of the data, the standard
errors and other statistics relating to uncertainty will be wrong when
we ignore dependence in the data.

To illustrate this, below we fit two models with the same mean structure to the NHANES data.  The first is a multiple regression model fit using **"ordinary least squares" (the default method for independent data)**.  The **second is fit using GEE**, which allows us to account for the dependence in the data.

In [ ]:
# Fit a linear model with OLS
model1 = sm.OLS.from_formula("BPXSY1 ~ RIDAGEYR + RIAGENDRx + BMXBMI + C(RIDRETH1)",
           data=da)
result1 = model1.fit()

# Fit a marginal linear model using GEE to handle dependent data
model2 = sm.GEE.from_formula("BPXSY1 ~ RIDAGEYR + RIAGENDRx + BMXBMI + C(RIDRETH1)",
           groups="group",
           cov_struct=sm.cov_struct.Exchangeable(), data=da)
result2 = model2.fit()

x = pd.DataFrame({"OLS_params": result1.params, "OLS_SE": result1.bse, "GEE_params": result2.params, "GEE_SE": result2.bse})

x = x[["OLS_params", "OLS_SE", "GEE_params", "GEE_SE"]]
print(x)

                   OLS_params    OLS_SE  GEE_params    GEE_SE
Intercept           91.736583  1.339378   92.168530  1.384309
RIAGENDRx[T.Male]    3.671294  0.453763    3.650245  0.454498
C(RIDRETH1)[T.2]     0.855488  0.819486    0.159296  0.767025
C(RIDRETH1)[T.3]    -1.796132  0.671954   -2.233280  0.760228
C(RIDRETH1)[T.4]     3.813314  0.732355    3.105654  0.881580
C(RIDRETH1)[T.5]    -0.455347  0.808948   -0.439831  0.813675
RIDAGEYR             0.478699  0.012901    0.474101  0.018493
BMXBMI               0.278015  0.033285    0.280205  0.038553


The OLS and GEE models produce very similar coefficient estimates, meaning the estimated effects of age, BMI, gender, and ethnicity are nearly the same. However, the GEE standard errors are generally larger because GEE accounts for clustering in the data. Since NHANES observations are not fully independent, the GEE results are more appropriate for inference. Therefore, we should trust the GEE standard errors more than the OLS standard errors.

GEE standard error is larger.

This matters because standard errors are used to calculate:

z\text{-values},\ p\text{-values},\ \text{confidence intervals}

So if the standard error is too small, the model may make results look more statistically significant than they really are.

## **Marginal logistic regression with dependent data**

Above we used GEE to fit marginal linear models in the presence of dependence.  GEE can also be used to fit any GLM in the presence of dependence.  We illustrate this using models relating smoking history to several demographic predictor variables.  These are the same models we fit in our previous notebook using GLM, which ignores the clustering.  Below we fit this same GLM again for comparison, then we fit the marginal model using GEE.  One thing to emphasize in doing this is that **the GLM and GEE are both legitimate estimators of the marginal mean structure here**. However GLM will not give correct standard errors, so everything derived from standard errors (e.g. confidence intervals and hypothesis tests) will not be correct.

In [ ]:
#smq:smoking status
# family=sm.families.Binomial() It tells Python:
# The outcome is binary, so use a logistic/binomial model.
# Because smq only has values 0 and 1, we should not use a normal linear model.

# Now the model or analysis can use education categories with readable names
da["DMDEDUC2x"] = da.DMDEDUC2.replace({1: "lt9", 2: "x9_11", 3: "HS", 4: "SomeCollege",
                                       5: "College", 7: np.nan, 9: np.nan})

# Fit a basic GLM
model1 = sm.GLM.from_formula("smq ~ RIDAGEYR + RIAGENDRx + C(DMDEDUC2x)",
           family=sm.families.Binomial(), data=da)
result1 = model1.fit()
result1.summary()

# Fit a marginal GLM using GEE
model2 = sm.GEE.from_formula("smq ~ RIDAGEYR + RIAGENDRx + C(DMDEDUC2x)",
           groups="group", family=sm.families.Binomial(),
           cov_struct=sm.cov_struct.Exchangeable(), data=da)
result2 = model2.fit(start_params=result1.params)

x = pd.DataFrame({"OLS_params": result1.params, "OLS_SE": result1.bse,
                  "GEE_params": result2.params, "GEE_SE": result2.bse})
x = x[["OLS_params", "OLS_SE", "GEE_params", "GEE_SE"]]
print(x)

                             OLS_params    OLS_SE  GEE_params    GEE_SE
Intercept                     -2.305999  0.114308   -2.249820  0.140567
RIAGENDRx[T.Male]              0.909597  0.060167    0.908682  0.062342
C(DMDEDUC2x)[T.HS]             0.943364  0.089663    0.887965  0.095397
C(DMDEDUC2x)[T.SomeCollege]    0.832227  0.084361    0.771636  0.104449
C(DMDEDUC2x)[T.lt9]            0.266228  0.109183    0.321784  0.141327
C(DMDEDUC2x)[T.x9_11]          1.098561  0.106697    1.062149  0.138401
RIDAGEYR                       0.018257  0.001725    0.017416  0.001803


As expected, the results show that the GLM and the GEE give very similar estimates for the regression parameters.  However the standard errors obtained using GEE are somewhat larger than those obtained using GLM.  This indicates that GLM understates the uncertainty in the estimated mean structure, which is a direct consequence of it ignoring the dependence structure.  The GEE results do not suffer from this weakness.

* GLM assumes each person is independent from every other person.
* GEE recognizes that people inside the same NHANES cluster may be similar.

To the extent that the GLM and GEE parameter estimates differ, this is due to GEE attempting to exploit the dependence structure to obtain more efficient (i.e. more accurate) estimates of the model parameters. Thus, in summary, we can view GEE as trying to accomplish three things above and beyond what we obtain from GLM:

* GEE gives us insight into the dependence structure of the data

* GEE uses the dependence structure to obtain meaningful standard errors of the estimated model parameters.

* GEE uses the dependence structure to estimate the model parameters more accurately

In contrast, GLM does not achieve the first point at all, and in terms of the second point, the GLM standard errors can be far too optimistic (i.e. too small) -- note that in the analysis we are pursuing here, even weak clustering (ICC around 0.02-0.04) modifies some of the standard errors by 10-40%.  Finally, with regard to the third  point, GEE should in general have an efficiency advantage over GLM, but GLM estimates remain "valid" and cannot be completely dismissed solely on this basis.

## **Multilevel models**

Multilevel modeling is a large topic, and some aspects of it are quite
advanced.  Here, we will explore one facet of multilevel modeling --
using it as an alternative way to accommodate dependence in clustered data.  In
this sense, multilevel modeling is an alternative to the marginal
regression analysis demonstrated above.

In the setting of linear regression, mutilevel models and marginal
models are similar in most ways (note that more substantial differences
between marginal and multilevel models emerge in the case of logistic
regression, and in other generalized linear or nonlinear models).
Multilevel models and marginal models estimate the same population
target, but represent this target in different ways, and utilize
different estimation procedures.

A multilevel model is usually expressed in terms of *random effects*.
These are variables that we do not observe, but that we can
nevertheless incorporate into a statistical model.  We cannot get into
all of the technical details here, but it is important to understand
that while these random effects are not observed, their presence can
be inferred through the data, as long as each random effect is modeled as
influencing at least two observations.

In the present setting, we are focusing only on dependence that arises
through a single level of clustering.  To put this in the context of
multilevel modeling, we can imagine that each cluster has a random
effect that is shared by all observations in that cluster.  For
example, if SBP tends to be around 0.5 units higher in one cluster,
then the random effect for that cluster would be 0.5, and it would add
to the predicted SBP for every observation in the cluster.

In [ ]:
# Fit a multilevel (mixed effects) model to handle dependent data
model = sm.MixedLM.from_formula("BPXSY1 ~ RIDAGEYR + RIAGENDRx + BMXBMI + C(RIDRETH1)",
           groups="group", data=da)
result = model.fit()
result.summary()

<class 'statsmodels.iolib.summary2.Summary'>
"""
           Mixed Linear Model Regression Results
============================================================
Model:             MixedLM  Dependent Variable:  BPXSY1     
No. Observations:  5102     Method:              REML       
No. Groups:        30       Scale:               256.6952   
Min. group size:   106      Log-Likelihood:      -21409.8702
Max. group size:   226      Converged:           Yes        
Mean group size:   170.1                                    
------------------------------------------------------------
                  Coef.  Std.Err.   z    P>|z| [0.025 0.975]
------------------------------------------------------------
Intercept         92.173    1.402 65.752 0.000 89.426 94.921
RIAGENDRx[T.Male]  3.650    0.452  8.084 0.000  2.765  4.535
C(RIDRETH1)[T.2]   0.153    0.887  0.172 0.863 -1.586  1.891
C(RIDRETH1)[T.3]  -2.238    0.758 -2.954 0.003 -3.723 -0.753
C(RIDRETH1)[T.4]   3.098    0.836  3.707 0.000  1.460  4.737
C(RIDRETH1)[T.5]  -0.439    0.878 -0.500 0.617 -2.161  1.282
RIDAGEYR           0.474    0.013 36.482 0.000  0.449  0.500
BMXBMI             0.280    0.033  8.404 0.000  0.215  0.346
group Var          3.615    0.085                           
============================================================

"""

\text{group variance} = 3.615 This means: Groups differ from each other in their average outcome.

**What does group variance = 3.615 mean?**

The model says that there is variation across groups.

To understand the typical difference between two randomly selected groups, they calculate:

\sqrt{2 \times 3.615} \approx 2.69

So:

Two randomly chosen groups would differ by about 2.69 units in the outcome, on average.

That is a meaningful difference. The text says this difference is comparable to:

* the difference between females and males, or
* about 6 years of aging.

So even though the group variance may look small, it has practical importance.

**ICC from the mixed model**

The ICC asks:

What proportion of total variation is due to differences between groups?

The formula is:

ICC = \frac{\text{group variance}}{\text{group variance} + \text{residual variance}}

Here:

ICC = \frac{3.615}{3.615 + 256.7}

ICC \approx 0.014

So:

ICC \approx 1.4\%

This means:

About 1.4% of the total variation in the outcome is due to differences between groups.


The mixed model estimates how much groups differ from each other. In this example, the group variance is 3.615, meaning there is some between-group variation. The ICC is about 0.014, so around 1.4% of the total variation is explained by group membership. This is similar to the ICC estimated by GEE. **Both GEE and mixed models handle clustered data, but mixed models also quantify the group-level variance directly.**

The **"variance structure parameters" are what distinguish a mixed model** from a marginal model. Here we only have one such parameter, which is the variance for groups, estimated to be 3.615. This means that if we were to choose two groups at random, their random effects would differ on average by around 2.69 (this is calculated as the square root of  2⋅3.615 ). This is a sizable shift, comparable to the difference between females and males, or to around 6 years of aging.

We have seen here that it least in this setting, the mixed modeling procedure accommodates dependence in the data, provides rigorous estimates of the strength of this dependence, and accounts for the dependence in both estimation and inference for the regression parameters. In this sense, the multilevel model has the same desirable properties as GEE (at least in this setting). In fact, each of these two methods is very useful and widely utilized. There are some settings where either GEE or multilevel modeling can be argued to have an advantage, but neither uniformly dominates the other.

Multilevel models can also be used to estimate ICC values. In the case of a model with one level, which is what we have here, the ICC is the variance of the grouping variable (3.615) divided by the sum of the variance of the grouping variable and the unexplained variance (256.7). Note that the unexplained variance is in upper part of the output, labeled scale. This ratio is around 0.014, which is very similar to the estimated ICC obtained using GEE.



### **Predicted random effects**

While the actual random effects in a multilevel model are never observable, we can predict them from the data.  The predicted random effects are commonly known as *BLUPs*, for "Best Linear Unbiased Predictors".

Examing the BLUPs is sometimes useful, although the emphasis in multilevel regression usually lies with the structural parameters that underlie the random effects, and not the random effects themselves.  In the NHANES analysis, for example, we could use the predicted random effects to quantify the uniqueness of each county relative to the mean.

The predicted random effects for the 30 groups (MVUs) in this analysis are shown below:

In [ ]:
#Level 2
result.random_effects

{np.int64(1191): group   -1.630976
 dtype: float64,
 np.int64(1192): group   -0.086162
 dtype: float64,
 np.int64(1201): group   -2.042661
 dtype: float64,
 np.int64(1202): group   -0.147472
 dtype: float64,
 np.int64(1211): group    0.280623
 dtype: float64,
 np.int64(1212): group    1.580732
 dtype: float64,
 np.int64(1221): group    0.283347
 dtype: float64,
 np.int64(1222): group    0.131512
 dtype: float64,
 np.int64(1231): group   -2.038171
 dtype: float64,
 np.int64(1232): group    0.617651
 dtype: float64,
 np.int64(1241): group    2.878488
 dtype: float64,
 np.int64(1242): group   -0.519364
 dtype: float64,
 np.int64(1251): group    2.064967
 dtype: float64,
 np.int64(1252): group    1.521281
 dtype: float64,
 np.int64(1261): group   -1.261975
 dtype: float64,
 np.int64(1262): group    0.980846
 dtype: float64,
 np.int64(1271): group    0.118031
 dtype: float64,
 np.int64(1272): group   -0.128397
 dtype: float64,
 np.int64(1281): group   -0.384862
 dtype: float64,
 np.int64(12

These are called BLUPs: Best Linear Unbiased Predictions.

Each number tells you how much that cluster is shifted above or below the overall average, after controlling for the fixed effects in the model.

Suppose the model predicts systolic blood pressure using:

\text{age} + \text{gender} + \text{ethnicity} + \text{BMI}

Then the random effect says:

After adjusting for age, gender, ethnicity, and BMI, does this cluster still have higher or lower blood pressure than expected?

**Result: Cluster 1241 is estimated to be about 2.88 blood pressure units higher than the average cluster, after controlling for the covariates.**

Final summary

result.random_effects shows how each level-2 group differs from the average group. Positive values mean the group is above average; negative values mean the group is below average. These differences are calculated after controlling for the fixed predictors, so unusually high or low BLUPs suggest that some unmeasured group-level characteristics may be influencing the outcome.


### **Random slopes**

Multilevel modeling is a broad framework that allows many different types of models to be specified and fit.  Here we are primarily focusing on the "random intercept models", that allow the data in each cluster to be shifted by a common amount, in order to account for stable confounders within clusters.  Above we found evidence that such clustering (non-independence) is present in the NHANES blood pressure data.  Next we consider a more subtle form of cluster effect, in which the slope for a specific covariate varies from cluster to cluster.  This is called a *random slopes model*.  To demonstrate, below we fit a model in which SBP has cluster-specific intercepts, and cluster-specific slopes for the age covariate.  That is, we ask whether the rate at which blood pressure increases with age might differ from one cluster to the next.

We fit two variations on this model.  In the first model, the cluster-specific intercepts and slopes are independent random variables.  That is, a cluster with unusually high SBP is no more or less likely to have unusually rapid increase of SBP with age.  Note that when working with random slopes, it is usually advisable to center any covariate which has a random slope.  This does not change the fundamental interpretation of the model, but it does often result in models that converge faster and more robustly.  Here we center within each cluster, although it is also common to center in the
entire dataset, rather than centering separately by cluster.

1. Random intercept model

A random intercept model allows each cluster to have a different baseline blood pressure.

So the lines are shifted up or down, but they are parallel.

Simple model idea:

SBP = \beta_0 + u_j + \beta_1 age

Where:

* \beta_0 = average intercept
* u_j = cluster-specific intercept
* \beta_1 = average age effect

Simple interpretation:

Some clusters have higher average blood pressure than others, but age affects blood pressure the same way in all clusters.

2. Random slope model

A random slope model allows the effect of age to differ by cluster.

Now the lines are not only shifted up or down. They can also have different steepness.

Simple model idea:

SBP = \beta_0 + u_j + (\beta_1 + v_j)age

Where:

* u_j = cluster-specific intercept
* v_j = cluster-specific age slope
* \beta_1 = average age effect

Simple interpretation:

In some clusters, blood pressure increases rapidly with age. In other clusters, it increases more slowly.

**Final summary**

A random intercept model allows each cluster to have a different baseline outcome.

A random slope model allows each cluster to have a different effect of a predictor.

In [ ]:
#da["age_cen"] do? It creates a new variable called age_cen.
# For each person, it calculates:
# age\_cen = age - \text{average age in that person’s group}
# So age is centered within each group.
da["age_cen"] = da.groupby("group").RIDAGEYR.transform(lambda x: x - x.mean())


#The 0+ means: Do not add another intercept in this variance component; only estimate the random slope for age_cen.
# Random effect only for the age slope, not for the intercept.

model = sm.MixedLM.from_formula("BPXSY1 ~ age_cen + RIAGENDRx + BMXBMI + C(RIDRETH1)",
           groups="group", vc_formula={"age_cen": "0+age_cen"}, data=da)
result = model.fit()
result.summary()

/usr/local/lib/python3.12/dist-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


<class 'statsmodels.iolib.summary2.Summary'>
"""
             Mixed Linear Model Regression Results
===============================================================
Model:              MixedLM   Dependent Variable:   BPXSY1     
No. Observations:   5102      Method:               REML       
No. Groups:         30        Scale:                263.7323   
Min. group size:    106       Log-Likelihood:       -21469.9240
Max. group size:    226       Converged:            Yes        
Mean group size:    170.1                                      
---------------------------------------------------------------
                   Coef.  Std.Err.   z    P>|z|  [0.025  0.975]
---------------------------------------------------------------
Intercept         115.207    1.209 95.265 0.000 112.836 117.577
RIAGENDRx[T.Male]   3.643    0.457  7.962 0.000   2.746   4.539
C(RIDRETH1)[T.2]    1.167    0.827  1.412 0.158  -0.453   2.787
C(RIDRETH1)[T.3]   -1.659    0.679 -2.444 0.015  -2.989  -0.328
C(RIDRETH1)[T.4]    3.610    0.739  4.884 0.000   2.161   5.058
C(RIDRETH1)[T.5]   -1.208    0.816 -1.480 0.139  -2.807   0.392
age_cen             0.467    0.018 26.235 0.000   0.432   0.502
BMXBMI              0.288    0.034  8.574 0.000   0.222   0.353
age_cen Var         0.004    0.000                             
===============================================================

"""

The coefficient for age_cen is: 0.467 means On average, systolic blood pressure increases by about 0.467 mm Hg per year of age, holding gender, BMI, and ethnicity constant.

**Random age slope variance**

The random slope variance is age\_cen \ Var = 0.004
*Variance is hard to interpret directly, so we take the square root*:

\sqrt{0.004} \approx 0.06

This means the standard deviation of the age slopes across clusters is about 0.06

**Simple interpretation**

The average age effect is 0.467 But each cluster can have a slightly different age effect.

So some clusters may have an age slope around:

0.467 + 0.06 = 0.527 In some clusters, SBP increases by about 0.527 mm Hg per year.

0.467 - 0.06 = 0.407 In other clusters, SBP increases by about 0.407 mm Hg per year.


The variance estimate is very small 0.004 There may be only **weak evidence that the age slope truly varies across clusters.**

We see that the estimated variance for random age slopes is 0.004, which translates to a standard deviation of 0.06.  That is, from one cluster to another, the age slopes fluctuate by $\pm 0.06-0.12$ (1-2 standard deviations).  These cluster-specific fluctuations are added/subtracted from the fixed effect for age, which is 0.467.  Thus, in some clusters SBP may increase by around $0.467 + 0.06 = 0.527$ mm Hg per year, while in other clusters SBP may increase by only around $0.467 - 0.06 = 0.407$ mm Hg per year.  Note also that the fitting algorithm produces a warning that the estimated variance parameter is close to the boundary.  In this case, however, the algorithm seems to have converged to a point just short of the boundary.

Next, we fit a model in which the cluster-specific intercepts and slopes are allowed to be correlated.

Next, we fit a model in which the **cluster-specific intercepts and slopes are allowed to be correlated.**

groups="group"

re_formula="1+age_cen"


Each group can deviate from the average intercept and from the average age slope.

Mathematically:



Where:

* \beta_0 = average intercept
* \beta_1 = average age effect
* u_{0j} = group-specific intercept shift
* u_{1j} = group-specific age slope shift

In [ ]:
model = sm.MixedLM.from_formula("BPXSY1 ~ age_cen + RIAGENDRx + BMXBMI + C(RIDRETH1)",
           groups="group", re_formula="1+age_cen", data=da)
result = model.fit()
result.summary()

/usr/local/lib/python3.12/dist-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


<class 'statsmodels.iolib.summary2.Summary'>
"""
              Mixed Linear Model Regression Results
=================================================================
Model:                MixedLM   Dependent Variable:   BPXSY1     
No. Observations:     5102      Method:               REML       
No. Groups:           30        Scale:                255.4451   
Min. group size:      106       Log-Likelihood:       -21413.6193
Max. group size:      226       Converged:            Yes        
Mean group size:      170.1                                      
-----------------------------------------------------------------
                     Coef.  Std.Err.   z    P>|z|  [0.025  0.975]
-----------------------------------------------------------------
Intercept           115.467    1.340 86.173 0.000 112.840 118.093
RIAGENDRx[T.Male]     3.662    0.451  8.121 0.000   2.778   4.546
C(RIDRETH1)[T.2]      0.023    0.898  0.025 0.980  -1.738   1.783
C(RIDRETH1)[T.3]     -2.251    0.778 -2.893 0.004  -3.775  -0.726
C(RIDRETH1)[T.4]      3.011    0.854  3.524 0.000   1.336   4.686
C(RIDRETH1)[T.5]     -0.585    0.893 -0.655 0.512  -2.336   1.165
age_cen               0.466    0.018 26.286 0.000   0.431   0.501
BMXBMI                0.283    0.033  8.497 0.000   0.218   0.349
group Var             8.655    0.169                             
group x age_cen Cov   0.119    0.004                             
age_cen Var           0.004    0.000                             
=================================================================

"""

**This is a multilevel linear model with random intercepts and random slopes for age.** and allows the group effect and the age effect to vary by cluster.

**Fix Effects:**
age\_cen = 0.466 Interpretation: On average, systolic blood pressure increases by about 0.466 mm Hg per year of age, holding gender, BMI, and ethnicity constant.

RIAGENDRx[T.Male] = 3.662 Interpretation: *Males have about 3.66 mm Hg higher systolic blood pressure than females*, holding other variables constant.

BMXBMI = 0.283 Interpretation: Each 1-unit increase in BMI is associated with about 0.283 mm Hg higher systolic blood pressure.


**Random effects**
These are the level-2, group-level parts.

**group Var** = 8.655 This means groups differ in their baseline blood pressure.

Some clusters have higher average blood pressure, and others have lower average blood pressure, after adjusting for age, gender, BMI, and ethnicity.

The standard deviation is:
\sqrt{8.655} \approx 2.94

So typical groups differ from the average group by about 2.94 mm Hg in baseline SBP.

**age_cen Var** = 0.004

This means the age slope varies across groups, but only slightly.
The standard deviation is:

\sqrt{0.004} \approx 0.063
So the age effect varies around:

0.466 \pm 0.063

Example:

* Some groups: 0.466 + 0.063 = 0.529
* Other groups: 0.466 - 0.063 = 0.403

So age increases blood pressure in all groups, but the strength of the age effect changes slightly by group.

**group x age_cen Cov = 0.119**

This is the covariance between the random intercept and random age slope.

Because it is positive, it suggests:

Groups with higher baseline blood pressure may also tend to have slightly stronger age effects.

**Main conclusion**

Blood pressure increases with age and BMI, and males have higher blood pressure than females. There are also differences between groups: some groups have higher baseline blood pressure, and the effect of age varies slightly across groups. The random intercept variation is more important than the random age-slope variation.

Again, we get a warning that the parameter estimates fall close to the boundary. The estimated correlation coefficient between random slopes and random intercepts is estimated to be $0.119/\sqrt{8.655\cdot 0.004}$, which is around $0.64$.  This indicates that the clusters with unusually high average SBP also tend to have SBP increasing faster with age.  Note
however that these structural parameters are only estimates, and due to the variance parameter falling close to the boundary, the estimates may not be particularly precise.